In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv('course_lead_scoring_2026.txt')
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [13]:
for col in df.columns:
    print(col, df[col].nunique(), df[col].unique(), '\n')

lead_source 5 ['organic_search' 'social_media' 'referral' 'paid_ads' nan 'events'] 

industry 6 ['technology' 'retail' 'manufacturing' 'education' 'healthcare' nan
 'finance'] 

employment_status 4 ['employed' 'student' nan 'unemployed' 'self_employed'] 

location 5 ['europe' 'south_america' nan 'north_america' 'africa' 'asia'] 

annual_income 4313 [88160. 72688. 44697. ... 53636. 69430. 81355.] 

number_of_courses_viewed 7 [3 4 2 1 0 5 6] 

interaction_count 14 [ 4  7  6  5  9  2  0  1  3 10  8 11 12 13] 

lead_score 99 [0.64 0.72 0.58 0.57 0.62 0.77 0.78 0.41 0.52 0.35 0.11 0.33 0.13 0.85
 0.55 0.56 0.5  0.4  0.47 0.73 0.76 0.74 0.53 0.6  0.32 0.34 0.83 0.37
 0.48 0.42 0.38 0.3  0.49 0.54 0.16 0.84 0.63 0.28 0.46 0.67 0.44 0.51
 0.71 0.81 0.66 0.39 0.75 0.22 0.01 0.21 0.31 0.45 0.23 0.43 0.68 0.59
 0.18 0.17 0.19 0.27 0.26 0.65 0.99 0.69 0.86 0.07 0.89 0.24 0.36 0.05
 0.92 0.14 0.06 0.97 0.7  0.93 0.87 0.88 0.15 0.09 0.61 0.29 0.79  nan
 0.2  0.04 0.8  0.03 0.98 0.25 0.91 0.82 0.96 0

In [32]:
categorical_features = []
for col in df.columns:
    if df[col].dtype == 'object':
        categorical_features.append(col)

print('categorical features: ', categorical_features)

numerical_features = [x for x in df.columns if x not in categorical_features+['converted']]
print('numerical features:', numerical_features)

categorical features:  ['lead_source', 'industry', 'employment_status', 'location']
numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [34]:
for col in categorical_features:
    df[col] = df[col].fillna('NA')

for col in numerical_features+['converted']:
    df[col] = df[col].fillna(0)

df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# Q1

In [35]:
df['industry'].mode()

0    technology
dtype: object

# Q2

In [44]:
n = len(numerical_features)
correlation_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        correlation_matrix[i][j] = df[numerical_features[i]].corr(df[numerical_features[j]])

print(numerical_features)
correlation_matrix

['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


array([[1.        , 0.16129999, 0.12284235, 0.22949591],
       [0.16129999, 1.        , 0.721609  , 0.75720424],
       [0.12284235, 0.721609  , 1.        , 0.9157458 ],
       [0.22949591, 0.75720424, 0.9157458 , 1.        ]])

# Q3

In [50]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
2155,events,finance,employed,south_america,71662.0,0,0,0.09,0
289,referral,NA,employed,asia,40220.0,2,5,0.55,1
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
3442,events,technology,employed,north_america,89316.0,1,5,0.00,1
...,...,...,...,...,...,...,...,...,...
3200,paid_ads,finance,employed,asia,81522.0,1,3,0.29,0
617,social_media,retail,employed,asia,33075.0,1,1,0.30,1
1992,organic_search,finance,employed,africa,71532.0,3,5,0.58,1
3301,social_media,retail,employed,asia,48836.0,0,0,0.03,0


In [53]:
y_train = df_train['converted']
y_val = df_val['converted']
y_text = df_test['converted']

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [56]:
from sklearn.metrics import mutual_info_score

for col in categorical_features:
    print(col, round(mutual_info_score(df_train[col], y_train),4))

lead_source 0.0343
industry 0.002
employment_status 0.019
location 0.001


# Q4

In [57]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [72]:
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37
2155,events,finance,employed,south_america,71662.0,0,0,0.09
289,referral,NA,employed,asia,40220.0,2,5,0.55
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45
3442,events,technology,employed,north_america,89316.0,1,5,0.00
...,...,...,...,...,...,...,...,...
3200,paid_ads,finance,employed,asia,81522.0,1,3,0.29
617,social_media,retail,employed,asia,33075.0,1,1,0.30
1992,organic_search,finance,employed,africa,71532.0,3,5,0.58
3301,social_media,retail,employed,asia,48836.0,0,0,0.03


In [77]:
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

X_train = dv.transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)
X_train[0]

array([6.598e+04, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 3.000e+00, 3.700e-01, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 2.000e+00])

In [79]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [80]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [85]:
y_pred = model.predict_proba(X_val)[:, 1]
converted = y_pred > 0.5
print('accuracy: ', round((y_val == converted).mean(), 2))

accuracy:  0.64


In [86]:
accuracy = model.score(X_val, y_val)
print(accuracy)

0.645


In [87]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(accuracy)

0.645


# Q5

In [88]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [94]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

    train_dict = df_train.drop(columns=[f]).to_dict(orient='records')
    val_dict = df_val.drop(columns=[f]).to_dict(orient='records')

    
    dv = DictVectorizer(sparse=False)
    dv.fit(train_dict)
    
    X_train = dv.transform(train_dict)
    X_val = dv.transform(val_dict)

    model.fit(X_train, y_train)

    accuracy = model.score(X_val, y_val)
    print(f, abs(accuracy - 0.645))

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


# Q6

In [ ]:
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

X_train = dv.transform(train_dict)
X_val = dv.transform(val_dict)

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

accuracy = model.score(X_val, y_val)
print(f, abs(accuracy - 0.645))